In [95]:
import pandas as  pd
import numpy as  np

In [96]:
#converting data xl to csv 
file_path = r"C:\Users\Nitin\Desktop\MAJOR PROJECT\code\smart-data-quality-optimizer\data\raw\online_retail_II.xlsx"

df = pd.read_excel(file_path)
 
save_path = r"C:\Users\Nitin\Desktop\MAJOR PROJECT\code\smart-data-quality-optimizer\data\raw\Actual_file.csv"

df.to_csv(save_path, index=False)

In [97]:
file_path = r"C:\Users\Nitin\Desktop\MAJOR PROJECT\code\smart-data-quality-optimizer\data\raw\Actual_file.csv"
df = pd.read_csv(file_path)

In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      525461 non-null  object 
 1   StockCode    525461 non-null  object 
 2   Description  522533 non-null  object 
 3   Quantity     525461 non-null  int64  
 4   InvoiceDate  525461 non-null  object 
 5   Price        525461 non-null  float64
 6   Customer ID  417534 non-null  float64
 7   Country      525461 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 32.1+ MB


In [99]:
#  original dataset
dfr = pd.read_csv(file_path)

#  Create working copy
dfw = dfr.copy(deep=True)

#  Check original data
print("raw Shape:", dfr.shape)

#  Work on the copy
print("Working Shape:", dfw.shape)
 

raw Shape: (525461, 8)
Working Shape: (525461, 8)


In [100]:
dfw.duplicated().sum()

np.int64(6865)

In [101]:
dfw = dfw.drop_duplicates()

In [102]:
dfw.duplicated().sum()

np.int64(0)

In [103]:
import numpy as np
# Missing values
dfw.count()
dfw.isnull().sum()
dfw = dfw.fillna(" ? ")

In [104]:
dfw.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [105]:
dfw.dtypes

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID     object
Country         object
dtype: object

In [106]:
# Convert InvoiceDate to datetime
dfw["InvoiceDate"] = pd.to_datetime(
    dfw["InvoiceDate"],
    errors="coerce"
)

 

In [107]:
dfw.isnull().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [108]:
print((dfw["Quantity"] < 0).sum())
print((dfw["Quantity"] == 0).sum())
print((dfw["Price"] < 0).sum())
print((dfw["Price"] == 0).sum())
print(dfw["StockCode"].nunique())
print(dfw["Country"].nunique())
print(dfw["Country"].nunique())
print(dfw["Description"].str.strip().eq("").sum())

12302
0
3
3681
4632
40
40
0


In [109]:
invoice_product_duplicates = dfw.duplicated(
    subset=["Invoice", "StockCode"],
    keep=False
)


In [110]:
invoice_product_duplicates.sum()

np.int64(12694)

In [111]:
print(
    dfw.loc[
        invoice_product_duplicates,
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price"]
    ].head(10)
)

    Invoice StockCode                         Description  Quantity  \
289  489488     22125        UNION JACK HOT WATER BOTTLE          2   
299  489488     22125        UNION JACK HOT WATER BOTTLE          1   
372  489517     21791  VINTAGE HEADS AND TAILS CARD GAME          6   
373  489517     21790                  VINTAGE SNAP CARDS         4   
392  489517     21791  VINTAGE HEADS AND TAILS CARD GAME          1   
393  489517     21790                  VINTAGE SNAP CARDS         1   
630  489529     22030              SWALLOWS GREETING CARD         1   
643  489529     22086     PAPER CHAIN KIT 50'S CHRISTMAS          1   
653  489529     22030              SWALLOWS GREETING CARD        12   
673  489529     22086     PAPER CHAIN KIT 50'S CHRISTMAS          3   

            InvoiceDate  Price  
289 2009-12-01 10:59:00   5.95  
299 2009-12-01 10:59:00   5.95  
372 2009-12-01 11:34:00   1.25  
373 2009-12-01 11:34:00   0.85  
392 2009-12-01 11:34:00   1.25  
393 2009-12-01 11:34

In [112]:
# {{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{{}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}}

In [113]:
# Group by invoice and product
grouped = dfw.groupby(
    ["Invoice", "StockCode"],
    dropna=False
)

# Count repeated invoice-product groups
repeated_groups = grouped.filter(
    lambda x: len(x) > 1
)

print("Rows in repeated groups:", len(repeated_groups))

# Check quantity and price variations
variation_report = grouped.agg(
    Rows=("Quantity", "size"),
    Unique_Quantities=("Quantity", "nunique"),
    Unique_Prices=("Price", "nunique")
)

variation_report = variation_report[
    variation_report["Rows"] > 1
]

print("\nRepeated groups:", len(variation_report))
print(
    "Groups with different quantities:",
    (variation_report["Unique_Quantities"] > 1).sum()
)
print(
    "Groups with different prices:",
    (variation_report["Unique_Prices"] > 1).sum()
)

Rows in repeated groups: 12694

Repeated groups: 6224
Groups with different quantities: 6138
Groups with different prices: 430


In [114]:
# Exact duplicate rows in the current dataset
exact_duplicates = dfw.duplicated(keep=False)

print("Rows involved in exact duplicates:", exact_duplicates.sum())

# Inspect exact duplicate records
print(
    dfw.loc[exact_duplicates]
       .sort_values(["Invoice", "StockCode"])
       .head(10)
)

Rows involved in exact duplicates: 0
Empty DataFrame
Columns: [Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country]
Index: []


In [115]:
# Numerical summary
print(dfw[["Quantity", "Price"]].describe())

# Detect statistical outliers using IQR
for col in ["Quantity", "Price"]:
    Q1 = dfw[col].quantile(0.25)
    Q3 = dfw[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = dfw[
        (dfw[col] < lower) |
        (dfw[col] > upper)
    ]

    print(f"\n{col} Outliers:", len(outliers))
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)

            Quantity          Price
count  518596.000000  518596.000000
mean       10.438152       4.716176
std       108.126593     147.090460
min     -9600.000000  -53594.360000
25%         1.000000       1.250000
50%         3.000000       2.100000
75%        10.000000       4.210000
max     19152.000000   25111.090000

Quantity Outliers: 57744
Lower Bound: -12.5
Upper Bound: 23.5

Price Outliers: 35078
Lower Bound: -3.1899999999999995
Upper Bound: 8.649999999999999


In [116]:
# Quantity and Price validation summary
print("Negative Quantity:",
      (dfw["Quantity"] < 0).sum())

print("Zero Quantity:",
      (dfw["Quantity"] == 0).sum())

print("Negative Price:",
      (dfw["Price"] < 0).sum())

print("Zero Price:",
      (dfw["Price"] == 0).sum())

# Inspect extreme records
print("\nExtreme Quantity Records:")
print(
    dfw.nlargest(10, "Quantity")[
        ["Invoice", "StockCode", "Quantity", "Price"]
    ]
)

print("\nExtreme Price Records:")
print(
    dfw.nlargest(10, "Price")[
        ["Invoice", "StockCode", "Quantity", "Price"]
    ]
)

Negative Quantity: 12302
Zero Quantity: 0
Negative Price: 3
Zero Price: 3681

Extreme Quantity Records:
       Invoice StockCode  Quantity  Price
90857   497946     37410     19152   0.10
127166  501534     21099     12960   0.10
127168  501534     21091     12960   0.10
127169  501534     21085     12744   0.10
127167  501534     21092     12480   0.10
192197  507637     84016     10200   0.00
135027  502269     21984     10000   0.25
135028  502269     21982     10000   0.25
135029  502269     21980     10000   0.25
135030  502269     21981     10000   0.25

Extreme Price Records:
        Invoice     StockCode  Quantity     Price
241824  C512770             M        -1  25111.09
241827   512771             M         1  25111.09
320581  C520667  BANK CHARGES        -1  18910.69
517953  C537630     AMAZONFEE        -1  13541.33
517955   537632     AMAZONFEE         1  13541.33
519294  C537651     AMAZONFEE        -1  13541.33
519170  C537644     AMAZONFEE        -1  13474.79
135012  C5

In [117]:
def generate_quality_report(df):
    report = {
        "Total Rows": len(df),
        "Total Columns": len(df.columns),
        "Missing Values": int(df.isnull().sum().sum()),
        "Exact Duplicate Rows": int(df.duplicated().sum()),
        "Invalid Dates": int(df["InvoiceDate"].isnull().sum()),
        "Negative Quantity": int((df["Quantity"] < 0).sum()),
        "Zero Price": int((df["Price"] == 0).sum()),
        "Negative Price": int((df["Price"] < 0).sum()),
        "Blank Descriptions": int(
            df["Description"].str.strip().eq("").sum()
        )
    }

    return pd.DataFrame(
        report.items(),
        columns=["Quality Check", "Affected Count"]
    )


quality_report = generate_quality_report(dfw)

print(quality_report.to_string(index=False))


       Quality Check  Affected Count
          Total Rows          518596
       Total Columns               8
      Missing Values               0
Exact Duplicate Rows               0
       Invalid Dates               0
   Negative Quantity           12302
          Zero Price            3681
      Negative Price               3
  Blank Descriptions               0


In [118]:
# Calculate issue percentages
total_rows = len(dfw)

negative_quantity_pct = (
    (dfw["Quantity"] < 0).sum() / total_rows
) * 100

zero_price_pct = (
    (dfw["Price"] == 0).sum() / total_rows
) * 100

negative_price_pct = (
    (dfw["Price"] < 0).sum() / total_rows
) * 100

print(f"Negative Quantity: {negative_quantity_pct:.2f}%")
print(f"Zero Price: {zero_price_pct:.2f}%")
print(f"Negative Price: {negative_price_pct:.4f}%")


Negative Quantity: 2.37%
Zero Price: 0.71%
Negative Price: 0.0006%


In [119]:
def generate_rule_report(df):
    total = len(df)

    rules = [
        {
            "Issue": "Negative Quantity",
            "Count": int((df["Quantity"] < 0).sum()),
            "Severity": "Review",
            "Action": "Check returns/cancellations"
        },
        {
            "Issue": "Zero Price",
            "Count": int((df["Price"] == 0).sum()),
            "Severity": "Review",
            "Action": "Verify transaction type"
        },
        {
            "Issue": "Negative Price",
            "Count": int((df["Price"] < 0).sum()),
            "Severity": "High",
            "Action": "Investigate price validity"
        },
        {
            "Issue": "Missing Values",
            "Count": int(df.isnull().sum().sum()),
            "Severity": "Review",
            "Action": "Check missing-value policy"
        },
        {
            "Issue": "Exact Duplicates",
            "Count": int(df.duplicated().sum()),
            "Severity": "Review",
            "Action": "Verify before removal"
        }
    ]

    report = pd.DataFrame(rules)
    report["Affected %"] = (
        report["Count"] / total * 100
    ).round(4)

    return report


rule_report = generate_rule_report(dfw)

print(rule_report.to_string(index=False))

            Issue  Count Severity                      Action  Affected %
Negative Quantity  12302   Review Check returns/cancellations      2.3722
       Zero Price   3681   Review     Verify transaction type      0.7098
   Negative Price      3     High  Investigate price validity      0.0006
   Missing Values      0   Review  Check missing-value policy      0.0000
 Exact Duplicates      0   Review       Verify before removal      0.0000


In [120]:
# 1. Negative quantity records
negative_qty = dfw[dfw["Quantity"] < 0]

# 2. Zero price records
zero_price = dfw[dfw["Price"] == 0]

# 3. Negative price records
negative_price = dfw[dfw["Price"] < 0]

print("NEGATIVE QUANTITY:")
print(negative_qty[
    ["Invoice", "StockCode", "Description", "Quantity", "Price"]
].head(5))

print("\nZERO PRICE:")
print(zero_price[
    ["Invoice", "StockCode", "Description", "Quantity", "Price"]
].head(5))

print("\nNEGATIVE PRICE:")
print(negative_price[
    ["Invoice", "StockCode", "Description", "Quantity", "Price"]
])


NEGATIVE QUANTITY:
     Invoice StockCode                    Description  Quantity  Price
178  C489449     22087       PAPER BUNTING WHITE LACE       -12   2.95
179  C489449    85206A   CREAM FELT EASTER EGG BASKET        -6   1.65
180  C489449     21895  POTTING SHED SOW 'N' GROW SET        -4   4.25
181  C489449     21896             POTTING SHED TWINE        -6   2.10
182  C489449     22083     PAPER CHAIN KIT RETRO SPOT       -12   2.95

ZERO PRICE:
     Invoice StockCode   Description  Quantity  Price
263   489464     21733  85123a mixed       -96    0.0
283   489463     71477         short      -240    0.0
284   489467    85123A   21733 mixed      -192    0.0
470   489521     21646            ?        -50    0.0
3114  489655     20683            ?        -44    0.0

NEGATIVE PRICE:
        Invoice StockCode      Description  Quantity     Price
179403  A506401         B  Adjust bad debt         1 -53594.36
276274  A516228         B  Adjust bad debt         1 -44031.79
403472  A528

In [121]:
def recommend_fixes(df):
    recommendations = []

    negative_qty = int((df["Quantity"] < 0).sum())
    zero_price = int((df["Price"] == 0).sum())
    negative_price = int((df["Price"] < 0).sum())

    if negative_qty > 0:
        recommendations.append({
            "Issue": "Negative Quantity",
            "Count": negative_qty,
            "Recommendation": "Check returns and credit notes",
            "Auto Fix": "No"
        })

    if zero_price > 0:
        recommendations.append({
            "Issue": "Zero Price",
            "Count": zero_price,
            "Recommendation": "Verify free items and missing prices",
            "Auto Fix": "No"
        })

    if negative_price > 0:
        recommendations.append({
            "Issue": "Negative Price",
            "Count": negative_price,
            "Recommendation": "Review accounting adjustments",
            "Auto Fix": "No"
        })

    return pd.DataFrame(recommendations)


recommendation_report = recommend_fixes(dfw)
print(recommendation_report.to_string(index=False))


            Issue  Count                       Recommendation Auto Fix
Negative Quantity  12302       Check returns and credit notes       No
       Zero Price   3681 Verify free items and missing prices       No
   Negative Price      3        Review accounting adjustments       No


In [122]:
from sklearn.ensemble import IsolationForest

# Features for anomaly detection
features = dfw[["Quantity", "Price"]].copy()

# Train AI model
model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(features)

# Predict anomalies
dfw["AI_Anomaly"] = model.predict(features)

# -1 = Anomaly, 1 = Normal
anomaly_count = (dfw["AI_Anomaly"] == -1).sum()

print("Total Records:", len(dfw))
print("AI Anomalies:", anomaly_count)
print("Anomaly Percentage:", round(anomaly_count / len(dfw) * 100, 2), "%")

Total Records: 518596
AI Anomalies: 25866
Anomaly Percentage: 4.99 %


In [123]:
# Extract AI-detected anomalies
ai_anomalies = dfw[dfw["AI_Anomaly"] == -1]

# Display 10 examples
print(
    ai_anomalies[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "AI_Anomaly"]
    ].head(10).to_string(index=False)
)

# Summary statistics
print("\nAnomaly Statistics:")
print(ai_anomalies[["Quantity", "Price"]].describe())

Invoice StockCode                         Description  Quantity  Price  AI_Anomaly
 489438     20711                     JUMBO BAG TOYS         60   1.30          -1
 489438    84031A  CHARLIE+LOLA RED HOT WATER BOTTLE         56   3.00          -1
 489438    84031B CHARLIE LOLA BLUE HOT WATER BOTTLE         56   3.00          -1
 489438    84032A  CHARLIE+LOLA PINK HOT WATER BOTTLE        60   1.90          -1
 489438    84519B     CARROT CHARLIE+LOLA COASTER SET        60   2.40          -1
 489438    85132A        CHARLIE + LOLA BISCUITS TINS        60   6.38          -1
 489438    85132C       CHARLIE AND LOLA FIGURES TINS        60   6.40          -1
 489438    85183A  CHARLIE & LOLA WASTEPAPER BIN BLUE        60   2.40          -1
 489438    85183B CHARLIE & LOLA WASTEPAPER BIN FLORA        60   2.40          -1
 489439      POST                             POSTAGE         3  18.00          -1

Anomaly Statistics:
           Quantity         Price
count  25866.000000  25866.00000

In [124]:
ai_anomalies = dfw[dfw["AI_Anomaly"] == -1]

print("AI anomalies:", len(ai_anomalies))

print(
    "Negative quantity:",
    (ai_anomalies["Quantity"] < 0).sum()
)

print(
    "Zero price:",
    (ai_anomalies["Price"] == 0).sum()
)

print(
    "Negative price:",
    (ai_anomalies["Price"] < 0).sum()
)

print(
    "Positive quantity and positive price:",
    (
        (ai_anomalies["Quantity"] > 0) &
        (ai_anomalies["Price"] > 0)
    ).sum()
)


AI anomalies: 25866
Negative quantity: 2659
Zero price: 1711
Negative price: 0
Positive quantity and positive price: 22878


In [125]:
# Rule-based issue flags
dfw["Rule_Negative_Quantity"] = dfw["Quantity"] < 0
dfw["Rule_Zero_Price"] = dfw["Price"] == 0
dfw["Rule_Negative_Price"] = dfw["Price"] < 0

# AI flag: -1 means anomaly
dfw["AI_Flag"] = dfw["AI_Anomaly"] == -1

# Combined report
combined_report = pd.DataFrame({
    "Detection Method": [
        "Rule: Negative Quantity",
        "Rule: Zero Price",
        "Rule: Negative Price",
        "AI: Isolation Forest"
    ],
    "Affected Records": [
        int(dfw["Rule_Negative_Quantity"].sum()),
        int(dfw["Rule_Zero_Price"].sum()),
        int(dfw["Rule_Negative_Price"].sum()),
        int(dfw["AI_Flag"].sum())
    ]
})

combined_report["Affected %"] = (
    combined_report["Affected Records"] / len(dfw) * 100
).round(2)

print(combined_report.to_string(index=False))

       Detection Method  Affected Records  Affected %
Rule: Negative Quantity             12302        2.37
       Rule: Zero Price              3681        0.71
   Rule: Negative Price                 3        0.00
   AI: Isolation Forest             25866        4.99


In [126]:
total_rows = len(dfw)

missing_rows = int(dfw.isnull().any(axis=1).sum())
duplicate_rows = int(dfw.duplicated().sum())

# Rows with at least one known rule violation
rule_issue_rows = int(
    (
        (dfw["Quantity"] < 0) |
        (dfw["Price"] <= 0)
    ).sum()
)

# Count rows affected by any of these checks
problem_rows = int(
    (
        dfw.isnull().any(axis=1) |
        dfw.duplicated() |
        (dfw["Quantity"] < 0) |
        (dfw["Price"] <= 0)
    ).sum()
)

quality_score = (1 - problem_rows / total_rows) * 100

print("Total Rows:", total_rows)
print("Rows with Missing Values:", missing_rows)
print("Exact Duplicate Rows:", duplicate_rows)
print("Rows with Rule Issues:", rule_issue_rows)
print("Rows with Any Checked Issue:", problem_rows)
print("Initial Data Quality Score:", round(quality_score, 2), "%")

Total Rows: 518596
Rows with Missing Values: 0
Exact Duplicate Rows: 0
Rows with Rule Issues: 13865
Rows with Any Checked Issue: 13865
Initial Data Quality Score: 97.33 %


In [127]:
# Original dataset vs working dataset
print("Original Rows:", len(dfr))
print("Working Rows:", len(dfw))

print("\nOriginal Exact Duplicates:", dfr.duplicated().sum())
print("Working Exact Duplicates:", dfw.duplicated().sum())

print("\nRows Removed:", len(dfr) - len(dfw))

Original Rows: 525461
Working Rows: 518596

Original Exact Duplicates: 6865
Working Exact Duplicates: 0

Rows Removed: 6865


In [128]:
import pandas as pd

audit_log = pd.DataFrame([{
    "Operation": "Exact Duplicate Removal",
    "Input Rows": len(dfr),
    "Duplicates Detected": int(dfr.duplicated().sum()),
    "Rows Removed": len(dfr) - len(dfw),
    "Output Rows": len(dfw),
    "Status": "Verified"
}])

print(audit_log.to_string(index=False))

              Operation  Input Rows  Duplicates Detected  Rows Removed  Output Rows   Status
Exact Duplicate Removal      525461                 6865          6865       518596 Verified


In [129]:
audit_log.to_csv(
    "data_quality_audit_log.csv",
    index=False
)

print("Audit log exported successfully!")


Audit log exported successfully!


In [130]:
# 1. Quality report
quality_report.to_csv("quality_report.csv", index=False)

# 2. Recommendations report
recommendation_report.to_csv(
    "recommendation_report.csv", index=False
)

# 3. AI anomaly records
ai_anomalies.to_csv("ai_anomalies.csv", index=False)

print("All reports exported successfully!")

All reports exported successfully!
